In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# Chargement
# ==========================================================

df = pd.read_csv("benchmark_results_qwenMath.csv")

df.columns = df.columns.str.strip()

# On enlève les réponses non parsées
df = df[df["parse_error"].isna()].copy()

print("="*70)
print(f"Nombre total d'exemples : {len(df)}")
print("="*70)

# ==========================================================
# 1. LABEL ACCURACY
# ==========================================================

label_accuracy = (
    df["ground_truth_label"] ==
    df["prediction_label"]
).mean()

summary = pd.DataFrame({
    "Metric": ["Label Accuracy"],
    "Score (%)": [round(label_accuracy * 100,2)]
})

print("\n==============================")
print("LABEL ACCURACY")
print("==============================")

display(summary)

# ==========================================================
# 2. MATRICE DE CONFUSION
# ==========================================================

confusion = pd.crosstab(
    df["ground_truth_label"],
    df["prediction_label"],
    rownames=["Ground Truth"],
    colnames=["Prediction"],
    margins=True
)

print("\n==============================")
print("CONFUSION MATRIX")
print("==============================")

display(confusion)

# --------- Visualisation ---------

plt.figure(figsize=(5,4))

plt.imshow(confusion.iloc[:-1,:-1])

plt.xticks(
    range(2),
    confusion.columns[:-1]
)

plt.yticks(
    range(2),
    confusion.index[:-1]
)

plt.xlabel("Prediction")
plt.ylabel("Ground Truth")
plt.title("Confusion Matrix")

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            confusion.iloc[i,j],
            ha="center",
            va="center",
            fontsize=12,
            color="black"
        )

plt.colorbar()

plt.show()

# ==========================================================
# 3. PREDICTION ERROR ACCURACY
# ==========================================================
# UNIQUEMENT quand :
# GT = incorrect
# ET
# Prediction = incorrect
# ==========================================================

incorrect_detected = df[
    (df["ground_truth_label"] == "incorrect") &
    (df["prediction_label"] == "incorrect")
].copy()

prediction_error_accuracy = (
    incorrect_detected["ground_truth_error"] ==
    incorrect_detected["prediction_error"]
).mean()

summary_error = pd.DataFrame({
    "Metric":[
        "Prediction Error Accuracy"
    ],
    "Score (%)":[
        round(prediction_error_accuracy*100,2)
    ],
    "Support":[
        len(incorrect_detected)
    ]
})

print("\n==============================")
print("PREDICTION ERROR ACCURACY")
print("==============================")

display(summary_error)

# ==========================================================
# 4. ACCURACY PAR TYPE D'ERREUR
# ==========================================================

results = []

for error in sorted(
    incorrect_detected["ground_truth_error"].dropna().unique()
):

    subset = incorrect_detected[
        incorrect_detected["ground_truth_error"] == error
    ]

    acc = (
        subset["ground_truth_error"] ==
        subset["prediction_error"]
    ).mean()

    results.append({
        "Error Type": error,
        "Support": len(subset),
        "Accuracy (%)": round(acc*100,2)
    })

results = pd.DataFrame(results)

results = results.sort_values(
    "Accuracy (%)",
    ascending=False
)

print("\n==============================")
print("ACCURACY PER ERROR TYPE")
print("==============================")

display(results)

# ==========================================================
# 5. VISUALISATION
# ==========================================================

plt.figure(figsize=(10,5))

plt.bar(
    results["Error Type"],
    results["Accuracy (%)"]
)

plt.ylim(0,100)

plt.ylabel("Accuracy (%)")

plt.xlabel("Error Type")

plt.title("Prediction Error Accuracy by Error Type")

plt.xticks(rotation=30, ha="right")

plt.grid(axis="y", alpha=0.3)

plt.show()